# 4. RecursiveCharacterTextSplitter 重点详解

来源：《尚硅谷-10-RAG.pdf》，第 31–37 页；对应课件 **2.3.4 ②**。页码均指 PDF 阅读器中的页序。

整理方式：原理保留关键逻辑；第 2 章示例精简，重点学习拆分器源码与递归拆分。
只记录到 2.5.2，不收录 2.5.3 案例。

**运行约定**：在 `D:/grow/langchain/7_RAG` 打开本文件，选择
`D:/conda/envs/langchain/python.exe` 对应的 Python 内核；代码单元按本文件顺序运行。
各 Notebook 的变量相互独立。使用 `len` 的示例以 Python 字符长度计数，不能据此推断 Token 数。

## 4.1 为什么优先掌握它

**先按自然边界尝试拆开；对仍然太长的片段，逐级换用更细的分隔符；再合并短片段。**
与只使用一个分隔符的 `CharacterTextSplitter` 相比，它能继续处理过长片段。

默认分隔符：`["\n\n", "\n", " ", ""]`，分别对应段落、换行、空格、单字符。
这是有优先级的候选列表，不是对整篇文章连续执行四次全量切分。
`keep_separator` 默认是 `True`，即放在后一个片段开头；与基类默认值不同。

| 配置 | 作用 |
| --- | --- |
| `chunk_size`、`chunk_overlap`、`length_function` | 继承基类的长度与合并规则 |
| `separators` | 自定义粗到细的边界，中文可增加句号、问号、逗号等 |
| `keep_separator="end"` | 希望句号跟随前一段时明确使用 `"end"` |
| `is_separator_regex` | 默认 `False`；普通标点按字面值匹配 |
| `add_start_index=True` | 通过 Document 入口查看原文中的字符起点 |

下面只保留三组有代表性的示例：原课件短字符串、中文分隔符对比、PDF 的完整加载与拆分。

## 4.2 源码与中文注释

节选自本地 `langchain-text-splitters==1.1.1` 的
`D:/conda/envs/langchain/Lib/site-packages/langchain_text_splitters/character.py`，
LangChain 项目，MIT 许可证。只省略英文 docstring、整行英文注释，执行语句保持原样。
`_split_text_with_regex()` 是同模块的内部辅助函数，负责按模式拆分并按配置保留分隔符；
`_merge_splits()` 在上一份笔记中已展开，不再重复。

### 初始化与公共入口

```python
def __init__(
    self,
    separators: list[str] | None = None,
    keep_separator: bool | Literal["start", "end"] = True,  # noqa: FBT001,FBT002
    is_separator_regex: bool = False,  # noqa: FBT001,FBT002
    **kwargs: Any,
) -> None:
    super().__init__(keep_separator=keep_separator, **kwargs)
    # 优先尝试自然边界，最后用空字符串降到字符粒度。
    self._separators = separators or ["\n\n", "\n", " ", ""]
    self._is_separator_regex = is_separator_regex
```

```python
def split_text(self, text: str) -> list[str]:
    # 公共入口只是把配置的分隔符列表交给内部递归方法。
    return self._split_text(text, self._separators)
```

### 递归核心 _split_text

```python
def _split_text(self, text: str, separators: list[str]) -> list[str]:
    final_chunks = []
    separator = separators[-1]
    new_separators = []
    # 在本层可用的分隔符中，选择第一个实际能匹配文本的分隔符。
    for i, s_ in enumerate(separators):
        separator_ = s_ if self._is_separator_regex else re.escape(s_)
        if not s_:
            separator = s_
            break
        if re.search(separator_, text):
            separator = s_
            # 只有更靠后的细粒度分隔符会传给下一层递归。
            new_separators = separators[i + 1 :]
            break

    separator_ = separator if self._is_separator_regex else re.escape(separator)
    splits = _split_text_with_regex(
        text, separator_, keep_separator=self._keep_separator
    )

    good_splits = []
    # 保留分隔符时，它已经在片段里；合并时不能再次插入。
    separator_ = "" if self._keep_separator else separator
    for s in splits:
        # 源码是严格小于；等于 chunk_size 的片段也进入下面分支。
        if self._length_function(s) < self._chunk_size:
            good_splits.append(s)
        else:
            # 先输出已收集的短片段，避免与后面的递归结果乱序。
            if good_splits:
                merged_text = self._merge_splits(good_splits, separator_)
                final_chunks.extend(merged_text)
                good_splits = []
            # 没有更细分隔符时直接保留；自定义配置没有空字符串可能留下超长块。
            if not new_separators:
                final_chunks.append(s)
            else:
                # 超长片段继续递归；这一步不使用 Embedding，也不进行语义相似度计算。
                other_info = self._split_text(s, new_separators)
                final_chunks.extend(other_info)
    # 先输出已收集的短片段，避免与后面的递归结果乱序。
    if good_splits:
        merged_text = self._merge_splits(good_splits, separator_)
        final_chunks.extend(merged_text)
    return final_chunks
```

### 沿一次调用看清递归

```text
文本 = AAA\n\nBBBBBBBBBBBB，chunk_size=6，overlap=0
  1. 本层选中 \n\n：得到 AAA 和带分隔符的长 B 片段
  2. AAA 是短片段，先合并输出
  3. 长 B 片段递归到后续的 \n、空格、空字符串
  4. 无法靠更粗边界拆小的 B，最终按字符细分，再合并到每块 6 字符
```

本层在处理长片段之前会先合并已有短片段，递归分支也有自己的合并窗口。
因此不能把最终结果理解成「全篇先摊平为单字符，再用一个滑动窗口切到底」。

## 4.3 示例一：保留课件的短字符串

同一个对象分别调用三种入口，比较返回类型。`chunk_size=10` 是为了看清细节，
不是实际检索任务的推荐块长。

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "LangChain框架特性\n\n"
    "多模型集成(GPT/Claude)\n记忆管理功能\n"
    "链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

chunks = text_splitter.split_text(text)
print("split_text →", type(chunks[0]).__name__, [len(c) for c in chunks])

paragraphs = text_splitter.create_documents([text], metadatas=[{"source": "demo.txt"}])
print("create_documents →", type(paragraphs[0]).__name__)
for para in paragraphs:
    print(repr(para.page_content), para.metadata)

docs = [Document(page_content=text, metadata={"source": "demo.txt", "page": 0})]
paragraphs = text_splitter.split_documents(docs)
print("split_documents 首块:", paragraphs[0])

split_text → str [10, 3, 9, 8, 6, 9, 10, 10, 3]
create_documents → Document
'LangChain框' {'source': 'demo.txt', 'start_index': 0}
'架特性' {'source': 'demo.txt', 'start_index': 10}
'多模型集成(GPT' {'source': 'demo.txt', 'start_index': 15}
'/Claude)' {'source': 'demo.txt', 'start_index': 24}
'记忆管理功能' {'source': 'demo.txt', 'start_index': 33}
'链式调用设计。文档' {'source': 'demo.txt', 'start_index': 40}
'分析场景示例：需要处' {'source': 'demo.txt', 'start_index': 49}
'理PDF/Word等' {'source': 'demo.txt', 'start_index': 59}
'格式。' {'source': 'demo.txt', 'start_index': 69}
split_documents 首块: page_content='LangChain框' metadata={'source': 'demo.txt', 'page': 0, 'start_index': 0}


## 4.4 示例二：中文标点与 fallback

中文通常没有英文式空格边界。加入标点后，可优先沿句子或分句边界拆分。
列表最后的 `""` 提供字符级兜底；如果长句没有任何可匹配标点，仍有办法继续细分。
设置 `keep_separator="end"`，让句号随前句保留。

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = "RAG先检索外部资料。再把资料交给模型。中文应注意句子边界，避免直接切断术语。"

configs = {
    "默认分隔符": ["\n\n", "\n", " ", ""],
    "中文分隔符": ["\n\n", "\n", "。", "！", "？", "……", "，", " ", ""],
}
for name, separators in configs.items():
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=18,
        chunk_overlap=4,
        separators=separators,
        keep_separator="end",
        length_function=len,
    )
    print(name)
    for chunk in text_splitter.split_text(text):
        print(len(chunk), repr(chunk))

# 故意没有最后的空字符串：不存在中文句号时，超长片段可能原样返回。
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10, chunk_overlap=0, separators=["。"]
)
print("没有字符级 fallback:", [len(c) for c in text_splitter.split_text("甲" * 25)])

默认分隔符
18 'RAG先检索外部资料。再把资料交给模'
18 '料交给模型。中文应注意句子边界，避免'
11 '界，避免直接切断术语。'
中文分隔符
11 'RAG先检索外部资料。'
9 '再把资料交给模型。'
10 '中文应注意句子边界，'
9 '避免直接切断术语。'
没有字符级 fallback: [25]


观察输出时不要只检查「长度是否接近上限」，还要看句子是否完整、术语是否被拆开、
是否出现太多碎块。中文标点配置只是改善边界，不会保证每个块都包含完整主题。

## 4.5 示例三：PyPDFLoader → split_documents

保留课件 `docs → text_splitter → paragraphs` 的步骤和命名，路径改为当前工程。
这里只读本地 PDF，输出少量块用于观察；所有页仍参与拆分。

In [8]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(Path("D:\\grow\\langchain\\assistant_agent\\asset\\load\\04-load.pdf")))
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=["\n\n", "\n", "。", "！", "？", "……", "，", " ", ""],
    keep_separator="end",
    length_function=len,
    add_start_index=True,
)
paragraphs = text_splitter.split_documents(docs)

print("原始页数:", len(docs), "拆分块数:", len(paragraphs))
for para in paragraphs[:3]:  # 只限制显示，不限制加载和拆分
    print("块长:", len(para.page_content))
    print(para.page_content)
    print({key: para.metadata[key] for key in ("source", "page", "start_index")})

原始页数: 1 拆分块数: 2
块长: 197
"他的车，他的命！ 他忽然想起来，一年，二年，至少有三四年；一滴汗，两滴汗，不
知道多少万滴汗，才挣出那辆车。从风里雨里的咬牙，从饭里茶里的自苦，才赚出那辆车。
那辆车是他的一切挣扎与困苦的总结果与报酬，像身经百战的武士的一颗徽章。……他老想
着远远的一辆车，可以使他自由，独立，像自己的手脚的那么一辆车。" 
 
"他吃，他喝，他嫖，他赌，他懒，他狡猾， 因为他没了心，他的心被人家摘了去。他
{'source': 'D:\\grow\\langchain\\assistant_agent\\asset\\load\\04-load.pdf', 'page': 0, 'start_index': 0}
块长: 130
只剩下那个高大的肉架子，等着溃烂，预备着到乱死岗子去。……体面的、要强的、好梦想
的、利己的、个人的、健壮的、伟大的祥子，不知陪着人家送了多少回殡；不知道何时何地
会埋起他自己来， 埋起这堕落的、 自私的、 不幸的、 社会病胎里的产儿， 个人主义的末路鬼！
"
{'source': 'D:\\grow\\langchain\\assistant_agent\\asset\\load\\04-load.pdf', 'page': 0, 'start_index': 198}


## 4.6 复习与参数选择

- `chunk_size` 小：检索粒度更细，但上下文容易断；大：上下文更多，也可能混入无关内容。
- `chunk_overlap` 增加上下文延续，也增加重复内容、向量数量和存储成本；不要只追求重叠量。
- 默认递归拆分按字符计数；真正受模型 Token 限制时，需要匹配模型的 Tokenizer。
- 长度不满上限可能来自边界、递归分支或两端空白清理，不应只看块长就认定出错。
- `start_index` 是每个原 Document 内的位置；PDF 中页码与页内起点要一起使用。

**对课件的必要澄清**：`keep_separator=True` 等价于 `"start"`，不是自动把句尾标点放在前块末尾。
分隔符耗尽后仍可能保留超长块，所以「递归拆分必然不超长」需要具体配置作为前提。

补充核对：[LangChain 官方递归拆分说明](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)。

## 笔记导航

1. [7_1_RAG原理与流程](7_1_RAG原理与流程.ipynb)
2. [7_2_Document与文档加载](7_2_Document与文档加载.ipynb)
3. [7_3_TextSplitter源码与Character切分](7_3_TextSplitter源码与Character切分.ipynb)
4. [7_4_RecursiveCharacterTextSplitter详解](7_4_RecursiveCharacterTextSplitter详解.ipynb)
5. [7_5_其他拆分方式速记](7_5_其他拆分方式速记.ipynb)
6. [7_6_Embedding与向量存储](7_6_Embedding与向量存储.ipynb)